In [2]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / "python"))

Project directory set to: E:\Admin\Documents\dermascan


In [3]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import cv2

from dataset_manager import DatasetManager
from pytorch_model_exporter import PytorchModelExporter
from preprocessing import image_filters_opencv

In [4]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

## Exporting the trained ResNet Models for Lesion Classification

In [12]:
# Load a sample input tensor for ONNX export
dataset_manager = DatasetManager(
    data_dir=str(proj_dir / "data")
)
dataset_manager.load_cleaned_metadata()

image_id = "ISIC_0033238"
image = cv2.imread(dataset_manager.df[dataset_manager.df['image_id'] == 'ISIC_0033238']['image_path'].iloc[0], cv2.IMREAD_COLOR)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image_gray_np = np.array(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY))

# Load sample images for each type
# RGB
rgb_path = proj_dir / "data" / "raw" / f"{image_id}.jpg"
if not rgb_path.exists():
    rgb_path = proj_dir / "data" / dataset_manager.df.loc[dataset_manager.df["image_id"] == image_id].iloc[0]['image_path']
rgb_img = cv2.cvtColor(cv2.imread(str(rgb_path)), cv2.COLOR_BGR2RGB)

# Gabor
image_gabor_cv = image_filters_opencv.gabor_filter(image_gray_np)
image_gabor_cv = np.transpose(image_gabor_cv, (1, 2, 0))

# GLCM
image_glcm_cv = image_filters_opencv.glcm_filter(image_gray_np)
image_glcm_cv = np.transpose(image_glcm_cv, (1, 2, 0))

model_map = {
    "resnet18rgb": {
        "model": models.resnet18(),
        "input_channels": 3,
        "output_channels": 2,
        "output_path": proj_dir / "flutter" / "assets" / "models" / "resnet18rgb.onnx",
        "model_path": Path(r"E:\Admin\Documents\dermascan\python\results\models\ResNet_25022026-195039_rgb.pt"),
        "sample_img": rgb_img
    },
    "resnet18gabor": {
        "model": models.resnet18(),
        "input_channels": 40,
        "output_channels": 2,
        "output_path": proj_dir / "flutter" / "assets" / "models" / "resnet18gabor.onnx",
        "model_path": Path(r"E:\Admin\Documents\dermascan\python\results\models\ResNet_06032026-181320_gabor_filter.pt"),
        "sample_img": image_gabor_cv
    },
    "resnet18glcm": {
        "model": models.resnet18(),
        "input_channels": 32,
        "output_channels": 2,
        "output_path": proj_dir / "flutter" / "assets" / "models" / "resnet18glcm.onnx",
        "model_path": Path(r"E:\Admin\Documents\dermascan\python\results\models\ResNet_16032026-115755_glcm_filter.pt"),
        "sample_img": image_glcm_cv
    }
}

### Defining the ResNet Model Architecture

In [13]:
# We define a helper to construct our model architecture
def create_resnet18_binary(input_channels: int):
    model = models.resnet18()
    
    # Modify input layer if not 3 channels
    if input_channels != 3:
        old_conv = model.conv1
        model.conv1 = nn.Conv2d(
            in_channels=input_channels,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias
        )
        with torch.no_grad():
            nn.init.kaiming_normal_(model.conv1.weight, mode='fan_out', nonlinearity='relu')

    # Modify output layer for 2 classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2)
    
    return model

def export_and_test_model(selected_model_type: str):
    model_info = model_map[selected_model_type]
    input_channels = model_info["input_channels"]
    output_channels = model_info["output_channels"]
    sample_img = model_info["sample_img"]
    model_path = model_info["model_path"]
    onnx_model_path = model_info["output_path"]
    
    exporter = PytorchModelExporter(input_channels=input_channels, output_channels=output_channels)
    
    img_tensor = exporter.load_sample_tensor(
        sample_img=sample_img,
        target_shape=(224, 224),
        normalize=True
    )

    print(f"[{selected_model_type}] Input tensor shape: {img_tensor.shape}")
    
    torch_model = exporter.build_torch_model(
        model_definition=create_resnet18_binary(input_channels),
        pretrained_weights=model_path,
        device=str(DEVICE)
    )

    onnx_model = exporter.export_torch_model(
        example_input=img_tensor,
        save_to=onnx_model_path
    )
    
    print(f"[{selected_model_type}] Model exported to {onnx_model_path}")
    
    distance_pct, torch_outputs, onnxruntime_outputs = exporter.compare_model_outputs(img_tensor, onnx_model_path)
    print(f"[{selected_model_type}] Distance between PyTorch and ONNX outputs: {distance_pct:.4f}%")

    def get_prediction(logits):
        exp_logits = np.exp(logits - np.max(logits))
        probs = exp_logits / np.sum(exp_logits)
        class_idx = np.argmax(probs)
        return ["Melanocytic Nevi (nv)", "Melanoma (mel)"][class_idx], probs[class_idx]
    
    pred_class, conf = get_prediction(onnxruntime_outputs[0])
    print(f"[{selected_model_type}] Predicted Class: {pred_class} with {conf:.2%} confidence")
    print("-" * 50)

### Exporting the RGB Model to ONNX

In [6]:
export_and_test_model("resnet18rgb")

[resnet18rgb] Input tensor shape: torch.Size([1, 3, 224, 224])
[resnet18rgb] Model exported to E:\Admin\Documents\dermascan\flutter\assets\models\resnet18rgb.onnx
[resnet18rgb] Distance between PyTorch and ONNX outputs: 0.0001%
[resnet18rgb] Predicted Class: Melanoma (mel) with 90.00% confidence
--------------------------------------------------


### Exporting the Gabor Model to ONNX

In [14]:
export_and_test_model("resnet18gabor")

[resnet18gabor] Input tensor shape: torch.Size([1, 40, 224, 224])
[resnet18gabor] Model exported to E:\Admin\Documents\dermascan\flutter\assets\models\resnet18gabor.onnx
[resnet18gabor] Distance between PyTorch and ONNX outputs: 0.0004%
[resnet18gabor] Predicted Class: Melanocytic Nevi (nv) with 64.81% confidence
--------------------------------------------------


### Exporting the GLCM Model to ONNX

In [15]:
export_and_test_model("resnet18glcm")

[resnet18glcm] Input tensor shape: torch.Size([1, 32, 224, 224])
[resnet18glcm] Model exported to E:\Admin\Documents\dermascan\flutter\assets\models\resnet18glcm.onnx
[resnet18glcm] Distance between PyTorch and ONNX outputs: 0.0017%
[resnet18glcm] Predicted Class: Melanoma (mel) with 55.30% confidence
--------------------------------------------------
